In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [3]:
import os
import sys

import io
from io import BytesIO
import csv

import google.auth
from google.cloud import bigquery
#from google.cloud import bigquery_storage

In [5]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./market-analysis-project-91130-9f9a036682b6.json"

credentials, project_id = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

bqclient = bigquery.Client(credentials=credentials, project=project_id,)

In [8]:

sql = f"""
select * from wook.amz_pdp_price_sales_gv
"""

df = bqclient.query(sql).to_dataframe()

In [10]:
df

,crawl_date,asin,rating,ratings_total,salesrank1,salesrank2,salesrank3,bw_price,revenue,units,gv
0,2023-08-17,B071YQG17N,4.6,19051.0,2932,3,18,292.64,585.28,2.0,117
1,2023-08-17,B01KGRAHJ0,4.7,4583.0,186693,257,None,173.11,1904.21,11.0,135
2,2023-08-17,B0C77678WZ,NaN,NaN,35517,143,None,NaN,NaN,NaN,<NA>
3,2023-08-17,B0B23RJVF1,4.1,93.0,364862,607,None,349.00,NaN,NaN,15
4,2023-08-17,B08FD786NX,4.3,27.0,206413,938,3087,72.95,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...
2636767,2025-06-16,B0CP1LR1PW,4.3,161087.0,None,None,None,81.72,NaN,NaN,<NA>
2636768,2025-06-16,B01AB68AC2,4.5,44285.0,None,None,None,148.79,NaN,NaN,<NA>
2636769,2025-06-16,B07DNLCW6W,4.6,2371.0,None,None,None,NaN,NaN,NaN,<NA>
2636770,2025-06-16,B0D2VMPB46,4.4,18534.0,None,None,None,93.58,NaN,NaN,<NA>


In [12]:
df.to_csv("amz_pdp_price_sales_gv_0617.csv", index=False)

### 데이터 탐색

In [14]:
df = pd.read_csv('amz_pdp_price_sales_gv_0617.csv')

In [16]:
df['crawl_date'] = pd.to_datetime(df['crawl_date'])
print(df['crawl_date'].min())
print(df['crawl_date'].max())
print(df['crawl_date'].max() - df['crawl_date'].min())

2023-08-17 00:00:00
2025-06-16 00:00:00
669 days 00:00:00


In [28]:
total_revenue_2025 = df[df['crawl_date'].dt.year == 2025]['revenue'].sum()
print(total_revenue_2025)

137071144.89


In [18]:
# asin별로 revenue 합산 후 상위 5개 추출
#top5_asin = df.groupby('asin')['revenue'].sum().sort_values(ascending=False).head(5).reset_index()

# 2025년 데이터 필터링 후 asin별 units 합계 계산 및 top 10 추출
top20_asin = (
    df[df['crawl_date'].dt.year == 2025]
    .groupby('asin')['units']
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

In [30]:
# 원본 데이터프레임에서 해당 asin만 필터링
top20_df = df[df['asin'].isin(top20_asin['asin'])].copy()

In [32]:
top20_df

,crawl_date,asin,rating,ratings_total,salesrank1,salesrank2,salesrank3,bw_price,revenue,units,gv
167594,2023-10-20,B0CJ4N6S6S,NaN,NaN,602.0,2.0,NaN,NaN,NaN,NaN,NaN
173781,2023-10-21,B0CJ4N6S6S,NaN,NaN,55177.0,82.0,NaN,NaN,NaN,NaN,NaN
186379,2023-10-19,B0CJ4N6S6S,NaN,NaN,36093.0,198.0,NaN,NaN,NaN,NaN,NaN
190955,2023-10-25,B0CKYSMM1J,NaN,NaN,46587.0,122.0,NaN,NaN,NaN,NaN,NaN
191342,2023-10-25,B0CKYY27YP,NaN,NaN,9080.0,39.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2636666,2025-06-16,B0CKZ1CK1H,4.3,161087.0,NaN,NaN,NaN,398.00,NaN,NaN,NaN
2636693,2025-06-16,B0CKYZ3B83,4.3,161087.0,NaN,NaN,NaN,237.59,NaN,NaN,NaN
2636749,2025-06-16,B0CKYYHD47,4.4,78245.0,NaN,NaN,NaN,305.95,NaN,NaN,NaN
2636755,2025-06-16,B0CSJMCXKB,4.4,9874.0,NaN,NaN,NaN,90.01,NaN,NaN,NaN


In [36]:
# 해당 컬럼들 중 하나라도 NaN이면 제거
df_cleaned = top20_df.dropna(subset=['rating', 'ratings_total', 'bw_price', 'units', 'gv'])
df_cleaned = df_cleaned.rename(columns={'ratings_total': 'ratings_cnt'})
#df_cleaned = df_cleaned.rename(columns={'ratings_total': 'ratings_cnt'}, inplace=True)

In [38]:
df_cleaned

,crawl_date,asin,rating,ratings_cnt,salesrank1,salesrank2,salesrank3,bw_price,revenue,units,gv
638789,2024-01-12,B0CKZ1CK1H,4.4,106587.0,56478.0,1534.0,NaN,349.00,349.00,1.0,277.0
638835,2024-01-12,B0CKYY27YP,4.4,106587.0,18941.0,8.0,NaN,198.91,198.88,1.0,68.0
638917,2024-01-12,B0CKYYL8B9,4.4,106587.0,1836649.0,4905.0,NaN,209.99,259.00,1.0,48.0
639754,2024-01-13,B0CKZ1CK1H,4.4,152048.0,253.0,3.0,NaN,349.00,2443.00,7.0,351.0
643564,2024-01-12,B0CKYZPPMJ,4.4,106587.0,1952729.0,80.0,NaN,151.12,151.11,1.0,167.0
...,...,...,...,...,...,...,...,...,...,...,...
2635436,2025-06-12,B0CKZ1RXKH,4.3,160994.0,86.0,1.0,NaN,249.00,27411.11,110.0,1981.0
2635470,2025-06-13,B0CKYSMM1J,4.3,30261.0,2266.0,10.0,NaN,115.99,11828.19,102.0,1107.0
2635571,2025-06-13,B0CJ4N6S6S,4.5,67270.0,2053.0,11.0,NaN,71.31,987.02,14.0,317.0
2635618,2025-06-13,B0CP1LR1PW,4.3,161021.0,85.0,1.0,NaN,81.72,11925.84,146.0,1042.0


In [40]:
# asin별 행 개수 세기
df_cleaned['asin'].value_counts().reset_index()
#asin_counts.columns = ['asin', 'row_count']

,asin,count
0,B0CKYZPPMJ,499
1,B0CKZ1RXKH,490
2,B0CKYY27YP,489
3,B0CKYZC93L,489
4,B0CKYZ3B83,486
5,B0CKZ1CK1H,483
6,B0CKYYL8B9,478
7,B0CKYZ4DJB,475
8,B0CKYZCVXK,474
9,B0CKYYYYYF,473


In [42]:
df_cleaned.to_csv("amz_pdp_price_sales_gv_top20_0617.csv", index=False)